# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (sockets)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
import findspark
findspark.init()

from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Examples on Structured Streaming",
                   master_url="spark://spark-master:7077")

su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 03:47:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Create a data stream from a local socket

### Install netcat utility

In [2]:
!apt-get update
!apt-get install -y netcat

Hit:1 http://ports.ubuntu.com/ubuntu-ports jammy InRelease
Get:2 http://ports.ubuntu.com/ubuntu-ports jammy-updates InRelease [128 kB]
Get:3 http://ports.ubuntu.com/ubuntu-ports jammy-backports InRelease [127 kB]
Get:4 http://ports.ubuntu.com/ubuntu-ports jammy-security InRelease [129 kB]
Get:5 http://ports.ubuntu.com/ubuntu-ports jammy-updates/restricted arm64 Packages [6,869 kB]
Get:6 http://ports.ubuntu.com/ubuntu-ports jammy-updates/universe arm64 Packages [1,667 kB]
Get:7 http://ports.ubuntu.com/ubuntu-ports jammy-updates/main arm64 Packages [3,952 kB]
Get:8 http://ports.ubuntu.com/ubuntu-ports jammy-backports/main arm64 Packages [83.5 kB]
Get:9 http://ports.ubuntu.com/ubuntu-ports jammy-backports/universe arm64 Packages [33.7 kB]
Get:10 http://ports.ubuntu.com/ubuntu-ports jammy-security/main arm64 Packages [3,634 kB]
Get:11 http://ports.ubuntu.com/ubuntu-ports jammy-security/universe arm64 Packages [1,363 kB]
Get:12 http://ports.ubuntu.com/ubuntu-ports jammy-security/restricted 

### Connect Spark to the socket

In [ ]:
import pyspark.sql.functions as F

# Create the remote connection
lines = su.spark.readStream \
            .format("socket") \
            .option("host", "localhost") \
            .option("port", 9999) \
            .load()

# Perform some transformations to the input data (word counter)
words = lines.select(F.explode(F.split(lines.value, " ")).alias("word"))
word_count = words.groupBy("word").count()

# Send transformed data to the Sink
query = word_count.writeStream \
            .outputMode("complete") \
            .format("console") \
            .start()
query.awaitTermination(300)


26/03/26 03:47:21 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.
26/03/26 03:47:22 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-0b98f0a3-e6c5-4c02-b113-09a6cd92776d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/03/26 03:47:22 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|Hello|    1|
+-----+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|Hello|    1|
|  gay|    1|
| Manu|    1|
+-----+-----+



In [1]:
su.spark.stop()

NameError: name 'su' is not defined